# Project Write Up

## <span style="color:darkred;">1. Introduction</span>

A core workflow in private credit involves distilling lengthy transaction documents into concise materials for investment committee (IC) review. These materials (typically a short written memo or the opening slides of an IC deck) summarize key terms, strengths and risks, and high-level rationale for the investment. The underlying information is usually contained in credit agreements and related legal documents that can be several hundred pages long, written in dense legal language, and require substantial manual effort to parse. Analysts must identify terms that are commercially relevant and interpret them through the lens of a credit investor.

While modern AI tools have begun to support well-defined financial tasks (e.g., LBO modeling) or legal workflows, few systems address this intersection: understanding legal documentation while reasoning like an investor about what constitutes a strong or weak credit opportunity. This project explores whether LLMs can meaningfully automate this workflow and how their performance can be evaluated systematically.  

The project has two primary goals:
1. Create a **memo generation pipeline** that ingests credit agreements scraped from public filings, extracts meaningful information, and generates a structured investment memo. Due to context limits, the output is scoped to the first one to two pages of an IC deck - an executive summary highlighting transaction details, key terms, and a concise assessment of strengths and risks.
2. Develop an **evaluation harness** to measure LLM performance across models (e.g., GPT-5, Claude Opus 4, Gemini 2.5 Pro) and across prompting strategies. The harness applies consistent metrics to assess whether techniques such as few-shot prompting, iterative refinement, chain-of-thought prompting, and more lead to systematic improvements.

For a live demo, see https://ai-for-finance-pbjgnzqdnz7ftrc3uabysv.streamlit.app/

Several avenues of model customization were initially considered, particularly supervised fine-tuning and reinforcement learning-based approaches, but were deprioritized for several reasons. Fine-tuning can yield strong results for narrow, well-labeled domains, but several limitations made it unsuitable as a first step: 
- **Lack of labeled data**: Although it was fairly straightforward to scrape a large corpus of credit agreements, creating a high-quality dataset of paired inputs and outputs (credit agreements → investment memos) would have required substantial manual effort.
- **Risk of overfitting**: With limited labeled data, a fine-tuned model might simply mimic the style of a small “gold standard” set rather than truly “learn” the investment rationale and be able to generalize across diverse credit agreements. A fine tuned model also needs to be re-tuned as new data is available, and / or when a new model is released.
- **Compute constraints**: Training (even at small scale) was not feasible within available resources.
- **Premature specialization**: Before customizing a model, it felt more valuable to build an end-to-end pipeline and establish objective criteria for evaluating quality.

Reinforcement learning presented similar obstacles (lack of reward signals, difficulty defining an environment, significant compute demands…). Given these limitations, a prompt engineering approach offered the highest return on time and allowed rapid iteration towards a working system.


The project therefore centers on building a **practical, evaluation-driven system**. The focus is on designing an end-to-end pipeline that works on real financial documents, establishing measurable criteria for high-quality memo generation, understanding how prompt design influences output quality, and identifying where current LLMs fall short when interpreting and reasoning over legal credit documents. This evaluation-first foundation hopefully provides a realistic view of current model capabilities and creates a clear path for future extensions.

## <span style="color:darkred;">2. Dataset Build </span>

### Data Acquisition

This project uses public credit agreements filed on the SEC’s EDGAR system by large public companies (2023 onward) as inputs for memo generation. These documents are well suited because they are publicly accessible, contain consistent legal and financial terms (e.g., interest rates, maturities, covenants), and follow broadly standard credit-agreement structures.

Data was retrieved using [EDGAR Full-Text Search queries](https://www.sec.gov/edgar/search/#/q=%2522Credit%2520Agreement%2522%2520AND%2520EX-10.1) for filings containing “Credit Agreement” AND “EX-10.1.” This reliably retrieves credit agreement exhibits across issuers. A simple JavaScript web scraper extracted document URLs and downloaded the corresponding HTML files. This process produced 500 agreements (499 unique), providing a sufficiently large corpus for experimentation without paid data sources. For downstream processing, documents were stored in JSONL format with source URLs and plain-text content.

### Data Cleaning

Although EDGAR documents are generally high quality, they contain artifacts (headers, footers, disclaimers,...) that can hinder extraction. A preprocessing pipeline transformed raw filings into clean, model-ready text. The pipeline reads URLs; downloads each page; converts the HTML to normalized plain text (removing scripts, styles and artifacts); writes one JSON line per document with source URL and text.

### Train-Test Split

A deterministic URL-based hash function partitioned documents into training and evaluation sets, ensuring reproducibility and preventing leakage. Fifty documents were used for training, with the remaining 449 held out for testing – after exploratory runs, iterating on more than 50 documents proved too costly and time intensive.

## <span style="color:darkred;">3. Evaluation Framework</span>


### Chosen Benchmark Model

For the scope of this project, a single model needed to be chosen to iterate on, so as to explore whether prompt optimizing techniques could improve that given model. While this choice could have been made arbitrarily, one of three frontier models was picked (Gemini 2.5 pro, Claude Sonnet 4, GPT-5) based on some exploratory runs. This was done by sampling 3 inputs from the training set, and manually inspecting how the three models’ outputs differed - what instructions can the model follow, where does it seem to struggle? This was done prior to building out a full evaluation harness, so as to help inform the evaluation metrics, and ensure that they accurately assessed what seemed to be the main pain points of model performance.

For these exploratory runs, a template was built of what a typical memo should look like, and a thoughtful prompt that an “average” person would spend a few minutes writing was used:

> <div style="font-size:70%; color:#666;">
>
> You are a Private Credit Analyst at an investment fund. Using the information contained in the attached credit agreement, draft a professional investment memorandum structured in three sections:
>
> 1. Executive Summary: Provide a concise overview that includes:
> - Date of the agreement  
> - Borrower / Company overview  
> - Brief description of the transaction (type, structure, counterparties)  
> - Purpose of the financing  
> - Brief company background and context
>
> 2. Investment Highlights & Risks: Present clear, bullet-pointed analysis from the perspective of an investor:
> - Key strengths / credit positives  
> - Principal risks and mitigating factors
>
> 3. Key Deal Terms Table: Include a well-formatted table listing:
> - Deal size  
> - Deal price  
> - Interest rate (and type, if applicable)  
> - Maturity date  
> - Payment frequency  
> - Key covenants or financial maintenance terms
>
> Instructions:
> - Use only factual information explicitly found in the attached credit agreement.  
> - If specific data points are not provided, write "N/A"—do not infer or fabricate details.  
> - Maintain a clear, concise, and professional tone suitable for an internal investment committee memo.  
> - Align the structure, level of detail, and tone with the attached template memo for reference.
>
> </div>


While this prompt was not perfectly optimized and included some redundancies, it was thought of as a good benchmark of what an individual might prompt into a chatbot, and chosen as a baseline.

Based on the exploratory 3-input run, it seemed that the general structure and tone was respected by all 3 models. While the content was largely similar across models for all 3 inputs, Claude Sonnet 4 performed significantly better on one input in identifying some risks that GPT-5 and Gemini 2.5 pro did not find (they seemed to rely solely on highlights that were more explicitly written and did not necessarily make sense, for example GPT-5 named a highlight “execution supported by signatures”, which is not a material investment highlight and more so a "bare minimum"). Gemini 2.5 pro and GPT-5 also had a handful of additional hallucinations compared to Claude, it is unclear if these were statistically significant. 

Claude Sonnet 4 was therefore chosen as the baseline model to iterate on. This was largely a high level judgement, and did not reflect a systematic, robust comparison of the three models which was made more difficult by limited compute/time. 

### Chosen Evaluation Metrics

Model performance is typically assessed along three dimensions: latency, cost, and functional correctness. Since this project evaluates prompting strategies on a small span of models, latency and cost were deprioritized and not incorporated directly into evaluation scores. They were tracked informally, particularly where prompting variants (e.g., refinement loops, batched vs. non-batched calls) produced noticeable differences, but the core evaluation focused on functional correctness, defined through several dimensions relevant to memo quality in a private credit context. Functional correctness in this context was split into four main components:

#### <span style="color:darkblue;">1. Accuracy</span>

**Definition**: Whether the memo includes any financial terms or statements that do not appear in the underlying credit agreement.

**Implementation**: Three LLMs (GPT-5, Claude-Sonnet-4, Gemini-2.5-Pro) answer a targeted yes/no question for each memo. The accuracy score is the share of judges answering “no hallucinations present.” A semantic or rule-based matching approach was initially considered but rejected for several reasons, including:
- Credit agreements vary in formatting/wording (for example, the interest rate could be defined as “interest rate”, “note rate”, or as a “margin” above a set “base rate”...), making it difficult to exhaustively enumerate what “key terms.” could be 
- Semantic-match pipelines are brittle to paraphrasing (e.g., 5.25% vs. 525 bps). 

LLM consensus was preferred as it allows for context-specific understanding (distinguishing key economic terms from ancillary text), handles synonyms and reformulations naturally, adapts across heterogeneous document structures, and reduces single-model bias via voting.

#### <span style="color:darkblue;">2. Completeness</span>

**Definition**: Whether the memo omits any key terms that a credit analyst would reasonably expect to see.


**Implementation**: Same three-model consensus framework as accuracy, with a yes/no question (counted as a score of 1 or 0) asking whether any important information appears to be missing relative to the source document. 

#### <span style="color:darkblue;">3. Quality</span>

**Definition**: Whether the memo is well-structured, clear, appropriately toned, and has a structure consistent with the template.

**Implementation**: LLM judges score four sub-dimensions. The overall score is the average of the three judges’ scores across these four submetrics.
- Clarity
- Tone 
- Length (concise but complete)
- Structure (alignment with the memo template)

Each sub-metric has its own prompt -- for example, the prompt for "tone" is the following: 

> <div style="font-size:80%; color:#666;">
>
> You are evaluating an investment memo for appropriate professional tone.
>
> MEMO: {memo}
>
> Goal: Assess whether the memo's tone is appropriate for presentation to an investment committee.
>
> Tone Criteria:
> - Professional formality: Language is formal and businesslike, appropriate for executive decision-makers  
> - Objective presentation: Presents information factually without emotional language or hype  
> - Balanced perspective: Acknowledges both strengths and risks without being overly promotional or pessimistic  
> - Financial sophistication: Uses appropriate financial terminology without being overly technical or simplistic  
> - Confidence without arrogance: Presents analysis authoritatively but remains measured
>
> Evaluate the memo's tone on a scale from 0–100:
> - 90–100: Perfectly appropriate for investment committee, professional and balanced  
> - 70–89: Generally appropriate with minor tone issues  
> - 50–69: Somewhat appropriate but has noticeable tone problems  
> - 30–49: Inappropriate tone in multiple sections  
> - 0–29: Significantly inappropriate tone throughout
>
> Output format: Provide ONLY a number from 0–100 as your score.
>
> SCORE: [number]
>
> </div>

#### <span style="color:darkblue;">4. Consistency</span>

**Definition**: Whether the memo contradicts itself (e.g., listing a term as both a strength and a weakness, or stating different interest rates in different sections).

**Implementation**: Each judge answers a binary yes/no question using a consistency-check prompt focusing on logical coherence within the memo. Measures of stability across repeated runs were considered, including “worst-at-k” (i.e. the worst score among k model runs), but were excluded due to time and compute constraints.

### Evaluation Harness 

To compare prompting strategies systematically, the project required a workflow capable of generating and evaluating many memos efficiently. The main constraints were cost and runtime: each run involved generating dozens of outputs, with each output being evaluated by three different LLM judges using several prompts for each judge. Running evaluations through standard synchronous API calls quickly proved impractical, with initial attempts taking several hours.

To address this, all generation and evaluation steps were migrated to batch API workflows, allowing requests to be parallelized at scale. This reduced typical end-to-end evaluation time for a full run (50 inputs) to a couple of minutes, provided that the evaluation prompts were simple one-turn queries (i.e., no refinement or iterative loops). Batch processing also reduced cost by taking advantage of discounted pricing and minimizing idle compute time.

The evaluation harness follows a consistent five-step pipeline:
1. **Memo Generation**: For a given prompt configuration, memos are generated by a chosen model. All generation requests are submitted as a single batch job for efficiency.
2. **Batch Completion and Retrieval**: The system polls for job completion and downloads all generated memos once the batch finishes.
3. **Multi-Model Evaluation**: Each memo is evaluated independently by three LLM judges using the evaluation prompts defined in Section 3.1. These evaluations are also submitted as (three) batch jobs to parallelize the workload.
4. **Polling and Result Extraction**: Once evaluations complete, results are retrieved and parsed into structured JSONL outputs.
5. **Aggregation and Summary Statistics**: For each memo, metric scores are averaged across judges. At the run level, the harness computes mean, median, minimum, maximum, and standard deviation across all memos and metrics. These aggregates enable comparative analysis across prompting strategies.

This harness provided a repeatable, scalable framework for testing prompt-engineering techniques under consistent conditions. It enabled rapid iteration across many runs and models, making it possible to attempt to isolate the impact of specific prompt changes on memo quality.

### Benchmark Performance

The table below outlines the benchmark performance for the project - the initial baseline prompt fed to Claude Sonnet 4 on the full 50-input training dataset. As outlined below, the baseline (average score) is 61, suggesting moderate performance with definite room for improvement. The large range of scores (from ~27 to ~86) and standard deviation of 15.12 indicated somewhat high inconsistency across samples, though it is unclear if this is driven by heterogeneity in the data quality or in model performance. Tone, clarity and length seemed to be quite strong, while the model seems to struggle with completeness issues (more than half of the model votes across the dataset indicated completeness issues) and with structure. 

<div style="display:flex; gap:20px; align-items:flex-start;">

<table>
  <thead>
    <tr><th colspan="2">Benchmark</th></tr>
  </thead>
  <tbody>
    <tr><td>Total Memos Evaluated</td><td>50</td></tr>
    <tr><td>Mean</td><td>61.28</td></tr>
    <tr><td>Median</td><td>63.58</td></tr>
    <tr><td>Min</td><td>26.85</td></tr>
    <tr><td>Max</td><td>85.77</td></tr>
    <tr><td>Std Dev</td><td>15.12</td></tr>
  </tbody>
</table>

<table>
  <thead>
    <tr><th>Evaluator</th><th>Mean</th><th>Median</th></tr>
  </thead>
  <tbody>
    <tr><td>Claude Sonnet 4</td><td>76.7</td><td>67.7</td></tr>
    <tr><td>GPT-5</td><td>53.3</td><td>45.0</td></tr>
    <tr><td>Gemini 2.5 Pro</td><td>53.8</td><td>45.6</td></tr>
  </tbody>
</table>

<table>
  <thead>
    <tr><th>Metric</th><th>Score</th></tr>
  </thead>
  <tbody>
    <tr><td>Tone</td><td>90.2</td></tr>
    <tr><td>Clarity</td><td>79.4</td></tr>
    <tr><td>Length</td><td>79.1</td></tr>
    <tr><td>Structure</td><td>43.8</td></tr>
    <tr><td>Accuracy (% no hallucinations)</td><td>72.7%</td></tr>
    <tr><td>Completeness (% complete)</td><td>38.7%</td></tr>
    <tr><td>Consistency (% no issues)</td><td>60.7%</td></tr>
  </tbody>
</table>

</div>

Taking a closer look at variability in evaluation scores by evaluator, there seems to be potential self-agreement bias, where Claude’s evaluation of itself ranked much higher both on a mean and a median basis than the other two evaluators. Potential ways to mitigate this could have been to weigh GPT-5 and Gemini 2.5 pro’s evaluations more heavily in the overall averaging. However, given how small dataset was, it was unclear if there was a true bias -- equal weighings were therefore kept.

A manual examination of the best scoring and worst scoring outputs for this run revealed that the best scoring output was indeed of high quality, and the worst scoring output represented an instance in which all three evaluators agreed on several of the memo’s shortcomings (that were true), leading to very poor scores because the averages were low. In both cases, the ratings generally made sense. This suggests that extreme scores could be more reliable, as they reflect full evaluator agreement, than “middle range’ scores that may just reflect models disagreeing rather than mediocre performance. 

## <span style="color:darkred;">4. Prompt Optimizing </span>

With the evaluation harness in place, a variety of prompt optimizing techniques were explored with the aim of improving model performance. These approaches were informed by several guides including those from [OpenAI](https://platform.openai.com/docs/guides/prompt-engineering), [Google](https://cloud.google.com/discover/what-is-prompt-engineering?hl=en), and [Anthropic](https://platform.claude.com/docs/en/build-with-claude/prompt-engineering/overview) on prompt engineering.

### Readily Available Prompt Optimizers

As a first set of experiments, the project evaluated several readily available prompt-optimization tools. These tools are designed to refine an initial “baseline prompt” by applying structured best practices, such as removing ambiguity, tightening instructions, clarifying output formats, and ensuring internal consistency. The goal of this section was to assess whether these off-the-shelf optimizers produced meaningful improvements when applied to this task.

#### <span style="color:darkblue;">Anthropic Prompt Generator</span>

Claude’s built-in prompt generator produced a more explicit and structured version of the baseline prompt, primarily by clarifying document boundaries and enforcing stricter section headers.

Original Excerpt: 
> <div style="font-size:80%; color:#666;">
>
> You are a Private Credit Analyst at an investment fund. Using the information contained in the attached credit agreement, draft a professional investment memorandum structured in three sections:
>
> </div>

Claude-revised Excerpt: 
> <div style="font-size:80%; color:#666;">
>
> You are acting as a Private Credit Analyst at an investment fund. You will be provided with a credit agreement document and a template memo for reference. Your task is to draft a professional investment memorandum based solely on the information contained in the credit agreement.*<br>
> *Here is the credit agreement document: <br>
> <credit_agreement> <br>
> {{CREDIT_AGREEMENT}}<br>
> </credit_agreement> <br>
> Here is the template memo for reference on structure, tone, and level of detail:<br>
> <template_memo> <br>
> {{TEMPLATE_MEMO}}<br>
> </template_memo><br>
> Your investment memorandum must be structured in exactly three sections:
>
> </div>

This resulted in modest improvements in the mean, minimum, and maximum scores, and a lower standard deviation. Overall, scores were higher for GPT-5 and Gemini, and lower for Claude, which could be an indication of higher consistency among evaluators/lower self agreement bias. There were slight drops in clarity, tone, length, and structure scores, though it is unclear if this was statistically significant. 

<div style="display:flex; gap:20px; align-items:flex-start; font-size:90%;">

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Metric</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Anthropic Prompt Generator</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align:left;">Mean Score</td><td style="text-align:center;">61.28</td><td style="text-align:center;">63.84</td><td style="text-align:center;">+2.56 pts</td>
    </tr>
    <tr>
      <td style="text-align:left;">Median Score</td><td style="text-align:center;">63.58</td><td style="text-align:center;">60.09</td><td style="text-align:center;">−3.49 pts</td>
    </tr>
    <tr>
      <td style="text-align:left;">Min Score</td><td style="text-align:center;">26.85</td><td style="text-align:center;">34.75</td><td style="text-align:center;">+7.9 pts</td>
    </tr>
    <tr>
      <td style="text-align:left;">Max Score</td><td style="text-align:center;">85.77</td><td style="text-align:center;">93.85</td><td style="text-align:center;">+8.08 pts</td>
    </tr>
    <tr>
      <td style="text-align:left;">Std Dev</td><td style="text-align:center;">15.12</td><td style="text-align:center;">13.44</td><td style="text-align:center;">−1.68 pts</td>
    </tr>
  </tbody>
</table>

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Evaluator</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Anthropic Prompt Generator</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align:left;">GPT-5</td><td style="text-align:center;">53.33</td><td style="text-align:center;">56.05</td><td style="text-align:center;">+2.72</td>
    </tr>
    <tr>
      <td style="text-align:left;">Claude Sonnet 4</td><td style="text-align:center;">76.67</td><td style="text-align:center;">74.99</td><td style="text-align:center;">−1.68</td>
    </tr>
    <tr>
      <td style="text-align:left;">Gemini 2.5 Pro</td><td style="text-align:center;">53.84</td><td style="text-align:center;">60.48</td><td style="text-align:center;">+6.64</td>
    </tr>
  </tbody>
</table>

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Metric</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Anthropic Prompt Generator</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align:left;">Tone</td><td style="text-align:center;">90.19</td><td style="text-align:center;">89.38</td><td style="text-align:center;">-0.81</td>
    </tr>
    <tr>
      <td style="text-align:left;">Clarity</td><td style="text-align:center;">79.37</td><td style="text-align:center;">73.82</td><td style="text-align:center;">-5.55</td>
    </tr>
    <tr>
      <td style="text-align:left;">Length</td><td style="text-align:center;">79.13</td><td style="text-align:center;">77.24</td><td style="text-align:center;">-1.89</td>
    </tr>
    <tr>
      <td style="text-align:left;">Structure</td><td style="text-align:center;">43.80</td><td style="text-align:center;">43.42</td><td style="text-align:center;">-0.38</td>
    </tr>
    <tr>
      <td style="text-align:left;">Accuracy (% no hallucinations)</td><td style="text-align:center;">72.7%</td><td style="text-align:center;">79.3%</td><td style="text-align:center;">+6.6%</td>
    </tr>
    <tr>
      <td style="text-align:left;">Completeness (% complete)</td><td style="text-align:center;">38.7%</td><td style="text-align:center;">40.3%</td><td style="text-align:center;">+1.6%</td>
    </tr>
    <tr>
      <td style="text-align:left;">Consistency (% no issues)</td><td style="text-align:center;">60.7%</td><td style="text-align:center;">64.7%</td><td style="text-align:center;">+4.0%</td>
    </tr>
  </tbody>
</table>

</div>

#### <span style="color:darkblue;">OpenAI Prompt Optimizer</span>

OpenAI’s prompt optimizer (provided in the OpenAI Cookbook) performs a similar refinement process, explicitly targeting common failure modes: contradictions within instructions, missing or unclear format specifications and inconsistencies between instructions and any included examples/templates – essentially disambiguating the prompt to the maximum extent possible. This optimizer refined the baseline prompt by dividing it into explicit sections with titles (“Role and Objective”, “Instructions”, “Memo Structure”, “Output Format”), and, similarly to Claude, providing even more explicit instructions (for example, replaced “Deal Size” with “Deal Size | Numeric amount with currency”).

<div style="display:flex; gap:20px; align-items:flex-start; font-size:90%;">

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Metric</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Anthropic Prompt Generator</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (OpenAI vs. Anthropic)</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (OpenAI vs. Benchmark)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left;">Mean Score</td><td style="text-align:center;">61.28</td><td style="text-align:center;">63.84</td><td style="text-align:center;">65.72</td><td style="text-align:center;">1.88</td><td style="text-align:center;">4.44</td></tr>
    <tr><td style="text-align:left;">Median Score</td><td style="text-align:center;">63.58</td><td style="text-align:center;">60.09</td><td style="text-align:center;">63.99</td><td style="text-align:center;">3.90</td><td style="text-align:center;">0.41</td></tr>
    <tr><td style="text-align:left;">Min Score</td><td style="text-align:center;">26.85</td><td style="text-align:center;">34.75</td><td style="text-align:center;">41.77</td><td style="text-align:center;">7.02</td><td style="text-align:center;">14.92</td></tr>
    <tr><td style="text-align:left;">Max Score</td><td style="text-align:center;">85.77</td><td style="text-align:center;">93.85</td><td style="text-align:center;">92.52</td><td style="text-align:center;">-1.33</td><td style="text-align:center;">6.75</td></tr>
    <tr><td style="text-align:left;">Std Dev</td><td style="text-align:center;">15.12</td><td style="text-align:center;">13.44</td><td style="text-align:center;">11.36</td><td style="text-align:center;">-2.08</td><td style="text-align:center;">-3.76</td></tr>
  </tbody>
</table>

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Evaluator</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Anthropic Prompt Generator</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (OpenAI vs. Anthropic)</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (OpenAI vs. Benchmark)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left;">GPT-5</td><td style="text-align:center;">53.33</td><td style="text-align:center;">56.05</td><td style="text-align:center;">57.26</td><td style="text-align:center;">1.21</td><td style="text-align:center;">3.93</td></tr>
    <tr><td style="text-align:left;">Claude Sonnet 4</td><td style="text-align:center;">76.67</td><td style="text-align:center;">74.99</td><td style="text-align:center;">77.78</td><td style="text-align:center;">2.79</td><td style="text-align:center;">1.11</td></tr>
    <tr><td style="text-align:left;">Gemini 2.5 Pro</td><td style="text-align:center;">53.84</td><td style="text-align:center;">60.48</td><td style="text-align:center;">62.14</td><td style="text-align:center;">1.66</td><td style="text-align:center;">8.30</td></tr>
  </tbody>
</table>

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Metric</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Anthropic Prompt Generator</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (OpenAI vs. Anthropic)</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (OpenAI vs. Benchmark)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left;">Tone</td><td style="text-align:center;">90.19</td><td style="text-align:center;">89.38</td><td style="text-align:center;">90.00</td><td style="text-align:center;">0.62</td><td style="text-align:center;">-0.19</td></tr>
    <tr><td style="text-align:left;">Clarity</td><td style="text-align:center;">79.37</td><td style="text-align:center;">73.82</td><td style="text-align:center;">82.61</td><td style="text-align:center;">8.79</td><td style="text-align:center;">3.24</td></tr>
    <tr><td style="text-align:left;">Length</td><td style="text-align:center;">79.13</td><td style="text-align:center;">77.24</td><td style="text-align:center;">76.40</td><td style="text-align:center;">-0.84</td><td style="text-align:center;">-2.73</td></tr>
    <tr><td style="text-align:left;">Structure</td><td style="text-align:center;">43.80</td><td style="text-align:center;">43.42</td><td style="text-align:center;">42.58</td><td style="text-align:center;">-0.84</td><td style="text-align:center;">-1.22</td></tr>
    <tr><td style="text-align:left;">Accuracy (% no hallucinations)</td><td style="text-align:center;">72.7%</td><td style="text-align:center;">79.3%</td><td style="text-align:center;">84.0%</td><td style="text-align:center;">4.7%</td><td style="text-align:center;">11.3%</td></tr>
    <tr><td style="text-align:left;">Completeness (% complete)</td><td style="text-align:center;">38.7%</td><td style="text-align:center;">40.3%</td><td style="text-align:center;">44.7%</td><td style="text-align:center;">4.4%</td><td style="text-align:center;">6.0%</td></tr>
    <tr><td style="text-align:left;">Consistency (% no issues)</td><td style="text-align:center;">60.7%</td><td style="text-align:center;">64.7%</td><td style="text-align:center;">61.3%</td><td style="text-align:center;">-3.4%</td><td style="text-align:center;">0.6%</td></tr>
  </tbody>
</table>

</div>

Generally, this resulted in a sizable improvement compared to the benchmark, with all metrics being generally on par with or better than the benchmark’s performance. Overall, accuracy increased 11 points compared to the benchmark, and almost 5 points compared to the Anthropic optimized prompt. Standard deviation across scores was also much lower, at 11.36 vs. 15.12 in the benchmark. This is considered the best performance so far.

#### <span style="color:darkblue;">Meta-Prompting</span>

Claude was prompted to suggest minimal edits to its own instructions, resulting in a shorter and more streamlined prompt (see below). While this improved mean scores slightly on the training set, it increased variance and hallucination rates relative to the OpenAI-optimized prompt, so this approach was not retained.

Meta-prompt: 
> <div style="font-size:80%; color:#666;">
>
> When asked to optimize prompts, give answers from your own perspective – explain what specific phrases could be added to, or deleted from, this prompt to more consistently elicit the desired behavior or prevent the undesired behavior. <br>
> Here is a prompt: [baseline prompt].<br>
> The desired behavior from this prompt is for the agent to write an effective, well-structured investment memo using all information provided in the source document but no other information; and to reason like a credit investor in terms of key highlights and risks for the transaction caused by the source document.<br>
> However, the agent sometimes names the same point as both a highlight and a risk, introduces outside information not present in the input, or omits relevant information.<br>
> While keeping as much of the existing prompt intact as possible, what minimal edits or additions would you make to ensure the agent more consistently addresses these shortcomings?
>
> </div>

Original Prompt Excerpt: 
> <div style="font-size:80%; color:#666;">
>
> Instructions:<br>
> - Use only factual information explicitly found in the attached credit agreement. Do not supplement with external knowledge about the industry, company, or market conditions -- even if you possess such information.<br>
> - Before including any statement in the Highlights & Risks section, verify it is directly supported by specific terms, clauses, or data in the agreement.<br>
> - If specific data points are not provided, write "N/A"—do not infer or fabricate details.<br>
> - Review the entire credit agreement systematically before drafting. Ensure all material financial terms, covenant packages, security arrangements, and structural features are captured in your analysis.<br>
> - Maintain a clear, concise, and professional tone suitable for an internal investment committee memo.<br>
> - Align the structure, level of detail, and tone with the attached template memo for reference.
>
> </div>

Claude-revised Prompt Excerpt: 
> <div style="font-size:80%; color:#666;">
>
> Instructions:<br>
> - Use only factual information explicitly found in the attached credit agreement.<br>
> - If specific data points are not provided, write "N/A"—do not infer or fabricate details.<br>
> - Maintain a clear, concise, and professional tone suitable for an internal investment committee memo.<br>
> - Align the structure, level of detail, and tone with the attached template memo for reference.
>
> </div>


<div style="display:flex; gap:20px; align-items:flex-start; font-size:90%;">

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Metric</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Anthropic Prompt Generator</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Meta-Prompted</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (MP vs. OpenAI)</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (MP vs. Benchmark)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left;">Mean Score</td><td style="text-align:center;">61.28</td><td style="text-align:center;">63.84</td><td style="text-align:center;">65.72</td><td style="text-align:center;">67.62</td><td style="text-align:center;">1.90</td><td style="text-align:center;">6.34</td></tr>
    <tr><td style="text-align:left;">Median Score</td><td style="text-align:center;">63.58</td><td style="text-align:center;">60.09</td><td style="text-align:center;">63.99</td><td style="text-align:center;">68.52</td><td style="text-align:center;">4.53</td><td style="text-align:center;">4.94</td></tr>
    <tr><td style="text-align:left;">Min Score</td><td style="text-align:center;">26.85</td><td style="text-align:center;">34.75</td><td style="text-align:center;">41.77</td><td style="text-align:center;">34.11</td><td style="text-align:center;">-7.66</td><td style="text-align:center;">7.26</td></tr>
    <tr><td style="text-align:left;">Max Score</td><td style="text-align:center;">85.77</td><td style="text-align:center;">93.85</td><td style="text-align:center;">92.52</td><td style="text-align:center;">94.06</td><td style="text-align:center;">1.54</td><td style="text-align:center;">8.29</td></tr>
    <tr><td style="text-align:left;">Std Dev</td><td style="text-align:center;">15.12</td><td style="text-align:center;">13.44</td><td style="text-align:center;">11.36</td><td style="text-align:center;">15.22</td><td style="text-align:center;">3.86</td><td style="text-align:center;">0.10</td></tr>
  </tbody>
</table>

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Evaluator</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Anthropic Prompt Generator</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Meta-Prompted</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (MP vs. OpenAI)</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (MP vs. Benchmark)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left;">GPT-5</td><td style="text-align:center;">53.33</td><td style="text-align:center;">56.05</td><td style="text-align:center;">57.26</td><td style="text-align:center;">57.03</td><td style="text-align:center;">-0.23</td><td style="text-align:center;">3.70</td></tr>
    <tr><td style="text-align:left;">Claude Sonnet 4</td><td style="text-align:center;">76.67</td><td style="text-align:center;">74.99</td><td style="text-align:center;">77.78</td><td style="text-align:center;">78.64</td><td style="text-align:center;">0.86</td><td style="text-align:center;">1.97</td></tr>
    <tr><td style="text-align:left;">Gemini 2.5 Pro</td><td style="text-align:center;">53.84</td><td style="text-align:center;">60.48</td><td style="text-align:center;">62.14</td><td style="text-align:center;">67.18</td><td style="text-align:center;">5.04</td><td style="text-align:center;">13.34</td></tr>
  </tbody>
</table>

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Metric</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Anthropic Prompt Generator</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Meta-Prompted</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (MP vs. OpenAI)</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (MP vs. Benchmark)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left;">Tone</td><td style="text-align:center;">90.19</td><td style="text-align:center;">89.38</td><td style="text-align:center;">90.00</td><td style="text-align:center;">90.98</td><td style="text-align:center;">0.98</td><td style="text-align:center;">0.79</td></tr>
    <tr><td style="text-align:left;">Clarity</td><td style="text-align:center;">79.37</td><td style="text-align:center;">73.82</td><td style="text-align:center;">82.61</td><td style="text-align:center;">81.00</td><td style="text-align:center;">-1.61</td><td style="text-align:center;">1.63</td></tr>
    <tr><td style="text-align:left;">Length</td><td style="text-align:center;">79.13</td><td style="text-align:center;">77.24</td><td style="text-align:center;">76.40</td><td style="text-align:center;">74.03</td><td style="text-align:center;">-2.37</td><td style="text-align:center;">-5.10</td></tr>
    <tr><td style="text-align:left;">Structure</td><td style="text-align:center;">43.80</td><td style="text-align:center;">43.42</td><td style="text-align:center;">42.58</td><td style="text-align:center;">44.08</td><td style="text-align:center;">1.50</td><td style="text-align:center;">0.28</td></tr>
    <tr><td style="text-align:left;">Accuracy (% no hallucinations)</td><td style="text-align:center;">72.7%</td><td style="text-align:center;">79.3%</td><td style="text-align:center;">84.0%</td><td style="text-align:center;">78.7%</td><td style="text-align:center;">-5.3%</td><td style="text-align:center;">6.0%</td></tr>
    <tr><td style="text-align:left;">Completeness (% complete)</td><td style="text-align:center;">38.7%</td><td style="text-align:center;">40.3%</td><td style="text-align:center;">44.7%</td><td style="text-align:center;">58.4%</td><td style="text-align:center;">13.7%</td><td style="text-align:center;">19.7%</td></tr>
    <tr><td style="text-align:left;">Consistency (% no issues)</td><td style="text-align:center;">60.7%</td><td style="text-align:center;">64.7%</td><td style="text-align:center;">61.3%</td><td style="text-align:center;">60.8%</td><td style="text-align:center;">-0.5%</td><td style="text-align:center;">0.1%</td></tr>
  </tbody>
</table>

</div>

For the next step, the OpenAI optimized prompt was used as the “improved” starting point, given its performance was generally on par or better than the benchmark and other two prompts, and hallucinations significantly lower, which is one of the most crucial metrics.

### Adding Context

Adding explicit business and audience context to the prompt did not improve performance and slightly degraded accuracy, tone, and structure. This suggests that the base prompt already provided sufficient task framing, and that additional narrative context may introduce ambiguity or encourage hallucination. This modification was not retained.

<div style="display:flex; gap:20px; align-items:flex-start; font-size:90%;">

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Metric</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer + Context</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (with vs. without context)</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (with context vs. Benchmark)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left;">Mean Score</td><td style="text-align:center;">61.28</td><td style="text-align:center;">65.72</td><td style="text-align:center;">65.51</td><td style="text-align:center;">-0.21</td><td style="text-align:center;">4.23</td></tr>
    <tr><td style="text-align:left;">Median Score</td><td style="text-align:center;">63.58</td><td style="text-align:center;">63.99</td><td style="text-align:center;">60.57</td><td style="text-align:center;">-3.42</td><td style="text-align:center;">-3.01</td></tr>
    <tr><td style="text-align:left;">Min Score</td><td style="text-align:center;">26.85</td><td style="text-align:center;">41.77</td><td style="text-align:center;">33.92</td><td style="text-align:center;">-7.85</td><td style="text-align:center;">7.07</td></tr>
    <tr><td style="text-align:left;">Max Score</td><td style="text-align:center;">85.77</td><td style="text-align:center;">92.52</td><td style="text-align:center;">94.31</td><td style="text-align:center;">1.79</td><td style="text-align:center;">8.54</td></tr>
    <tr><td style="text-align:left;">Std Dev</td><td style="text-align:center;">15.12</td><td style="text-align:center;">11.36</td><td style="text-align:center;">15.72</td><td style="text-align:center;">4.36</td><td style="text-align:center;">0.60</td></tr>
  </tbody>
</table>

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Evaluator</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer + Context</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (with vs. without context)</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (with context vs. Benchmark)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left;">GPT-5</td><td style="text-align:center;">53.33</td><td style="text-align:center;">57.26</td><td style="text-align:center;">54.72</td><td style="text-align:center;">-2.54</td><td style="text-align:center;">1.39</td></tr>
    <tr><td style="text-align:left;">Claude Sonnet 4</td><td style="text-align:center;">76.67</td><td style="text-align:center;">77.78</td><td style="text-align:center;">77.42</td><td style="text-align:center;">-0.36</td><td style="text-align:center;">0.75</td></tr>
    <tr><td style="text-align:left;">Gemini 2.5 Pro</td><td style="text-align:center;">53.84</td><td style="text-align:center;">62.14</td><td style="text-align:center;">64.34</td><td style="text-align:center;">2.20</td><td style="text-align:center;">10.50</td></tr>
  </tbody>
</table>

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Metric</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer + Context</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (with vs. without context)</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (with context vs. Benchmark)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left;">Tone</td><td style="text-align:center;">90.19</td><td style="text-align:center;">90.00</td><td style="text-align:center;">80.69</td><td style="text-align:center;">-9.31</td><td style="text-align:center;">-9.50</td></tr>
    <tr><td style="text-align:left;">Clarity</td><td style="text-align:center;">79.37</td><td style="text-align:center;">82.61</td><td style="text-align:center;">90.21</td><td style="text-align:center;">7.60</td><td style="text-align:center;">10.84</td></tr>
    <tr><td style="text-align:left;">Length</td><td style="text-align:center;">79.13</td><td style="text-align:center;">76.40</td><td style="text-align:center;">77.40</td><td style="text-align:center;">1.00</td><td style="text-align:center;">-1.73</td></tr>
    <tr><td style="text-align:left;">Structure</td><td style="text-align:center;">43.80</td><td style="text-align:center;">42.58</td><td style="text-align:center;">42.53</td><td style="text-align:center;">-0.05</td><td style="text-align:center;">-1.27</td></tr>
    <tr><td style="text-align:left;">Accuracy (% no hallucinations)</td><td style="text-align:center;">72.7%</td><td style="text-align:center;">84.0%</td><td style="text-align:center;">78.7%</td><td style="text-align:center;">-5.3%</td><td style="text-align:center;">6.0%</td></tr>
    <tr><td style="text-align:left;">Completeness (% complete)</td><td style="text-align:center;">38.7%</td><td style="text-align:center;">44.7%</td><td style="text-align:center;">49.3%</td><td style="text-align:center;">4.6%</td><td style="text-align:center;">10.6%</td></tr>
    <tr><td style="text-align:left;">Consistency (% no issues)</td><td style="text-align:center;">60.7%</td><td style="text-align:center;">61.3%</td><td style="text-align:center;">61.3%</td><td style="text-align:center;">0.0%</td><td style="text-align:center;">0.6%</td></tr>
  </tbody>
</table>

</div>

**Note on RAG:** Retrieval-augmented generation is a common technique for enhancing performance by supplying additional relevant context, typically via vector search or built-in retrieval tools. Although RAG is useful when documents are large or must be retrieved dynamically, it was not relevant for this project since input documents were already provided directly to the model (all information needed was provided directly, so external information would not have helped) and document sizes fell comfortably within model context limits, making chunking, retrieval, or external indexing unnecessary. The primary challenge here was not retrieval but interpreting and structuring the legal and financial content into a coherent investment memo.

### Few-Shot Examples

Another prompt-optimization strategy involved incorporating few-shot examples; explicit input-output pairs demonstrating what a high-quality investment memo should look like. Three input-output examples were hand-crafted using randomly selected documents from the dataset that were not a part of the training set – these examples were viewed as relevant as they reflected real credit agreement structures the model could realistically encounter. The selected agreements varied in complexity and completeness, with some containing only limited information (e.g. maturity date missing from the source document), which was viewed as a potentially useful way to reduce hallucinations, since the model must learn to produce a concise memo grounded strictly in the available text. These few-shot examples were attached to the OpenAI-optimized prompt (without the additional context sentence).

<div style="display:flex; gap:20px; align-items:flex-start; font-size:90%;">

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Metric</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer + FS Examples</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (with vs. without FS)</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (with FS vs. Benchmark)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left;">Mean Score</td><td style="text-align:center;">61.28</td><td style="text-align:center;">65.72</td><td style="text-align:center;">68.96</td><td style="text-align:center;">3.24</td><td style="text-align:center;">7.68</td></tr>
    <tr><td style="text-align:left;">Median Score</td><td style="text-align:center;">63.58</td><td style="text-align:center;">63.99</td><td style="text-align:center;">68.96</td><td style="text-align:center;">4.97</td><td style="text-align:center;">5.38</td></tr>
    <tr><td style="text-align:left;">Min Score</td><td style="text-align:center;">26.85</td><td style="text-align:center;">41.77</td><td style="text-align:center;">43.94</td><td style="text-align:center;">2.17</td><td style="text-align:center;">17.09</td></tr>
    <tr><td style="text-align:left;">Max Score</td><td style="text-align:center;">85.77</td><td style="text-align:center;">92.52</td><td style="text-align:center;">92.81</td><td style="text-align:center;">0.29</td><td style="text-align:center;">7.04</td></tr>
    <tr><td style="text-align:left;">Std Dev</td><td style="text-align:center;">15.12</td><td style="text-align:center;">11.36</td><td style="text-align:center;">11.87</td><td style="text-align:center;">0.51</td><td style="text-align:center;">-3.25</td></tr>
  </tbody>
</table>

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Evaluator</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer + FS Examples</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (with vs. without FS)</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (with FS vs. Benchmark)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left;">GPT-5</td><td style="text-align:center;">53.33</td><td style="text-align:center;">57.26</td><td style="text-align:center;">55.43</td><td style="text-align:center;">-1.83</td><td style="text-align:center;">2.10</td></tr>
    <tr><td style="text-align:left;">Claude Sonnet 4</td><td style="text-align:center;">76.67</td><td style="text-align:center;">77.78</td><td style="text-align:center;">84.06</td><td style="text-align:center;">6.28</td><td style="text-align:center;">7.39</td></tr>
    <tr><td style="text-align:left;">Gemini 2.5 Pro</td><td style="text-align:center;">53.84</td><td style="text-align:center;">62.14</td><td style="text-align:center;">67.37</td><td style="text-align:center;">5.23</td><td style="text-align:center;">13.53</td></tr>
  </tbody>
</table>

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Metric</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer + FS Examples</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (with vs. without FS)</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (with FS vs. Benchmark)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left;">Tone</td><td style="text-align:center;">90.19</td><td style="text-align:center;">90.00</td><td style="text-align:center;">89.73</td><td style="text-align:center;">-0.27</td><td style="text-align:center;">-0.46</td></tr>
    <tr><td style="text-align:left;">Clarity</td><td style="text-align:center;">79.37</td><td style="text-align:center;">82.61</td><td style="text-align:center;">85.01</td><td style="text-align:center;">2.40</td><td style="text-align:center;">5.64</td></tr>
    <tr><td style="text-align:left;">Length</td><td style="text-align:center;">79.13</td><td style="text-align:center;">76.40</td><td style="text-align:center;">82.39</td><td style="text-align:center;">5.99</td><td style="text-align:center;">3.26</td></tr>
    <tr><td style="text-align:left;">Structure</td><td style="text-align:center;">43.80</td><td style="text-align:center;">42.58</td><td style="text-align:center;">40.82</td><td style="text-align:center;">-1.76</td><td style="text-align:center;">-2.98</td></tr>
    <tr><td style="text-align:left;">Accuracy (% no hallucinations)</td><td style="text-align:center;">72.7%</td><td style="text-align:center;">84.0%</td><td style="text-align:center;">78.7%</td><td style="text-align:center;">-5.3%</td><td style="text-align:center;">6.0%</td></tr>
    <tr><td style="text-align:left;">Completeness (% complete)</td><td style="text-align:center;">38.7%</td><td style="text-align:center;">44.7%</td><td style="text-align:center;">62.7%</td><td style="text-align:center;">18.0%</td><td style="text-align:center;">24.0%</td></tr>
    <tr><td style="text-align:left;">Consistency (% no issues)</td><td style="text-align:center;">60.7%</td><td style="text-align:center;">61.3%</td><td style="text-align:center;">60.0%</td><td style="text-align:center;">-1.3%</td><td style="text-align:center;">-0.7%</td></tr>
  </tbody>
</table>

</div>

This technique yielded better results: increases in mean, median, min, max, and standard deviation, along with a very meaningful (18 point) increase in completeness. Hallucinations were 5 points higher, indicating some potential confusion introduced by the few shot examples. However, given the meaningful increases in overall scores, this technique was retained in the system.

### Chain-of-Thought

Another prompting technique explored was chain-of-thought (CoT) prompting, which encourages the model to reason through the document step-by-step before producing its final memo. This was simply done by adding a short sentence in the “Role and Objective” section: “Think step by step”. This sentence was added to the OpenAI-optimized prompt, with few shot examples (as this was the best combination so far).

<div style="display:flex; gap:20px; align-items:flex-start; font-size:90%;">

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Metric</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer + FS Examples</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer + FS + CoT</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (FS: with vs. without CoT)</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (FS + CoT vs. Benchmark)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left;">Mean Score</td><td style="text-align:center;">61.28</td><td style="text-align:center;">68.96</td><td style="text-align:center;">72.19</td><td style="text-align:center;">3.23</td><td style="text-align:center;">10.91</td></tr>
    <tr><td style="text-align:left;">Median Score</td><td style="text-align:center;">63.58</td><td style="text-align:center;">68.96</td><td style="text-align:center;">76.51</td><td style="text-align:center;">7.55</td><td style="text-align:center;">12.93</td></tr>
    <tr><td style="text-align:left;">Min Score</td><td style="text-align:center;">26.85</td><td style="text-align:center;">43.94</td><td style="text-align:center;">35.27</td><td style="text-align:center;">-8.67</td><td style="text-align:center;">8.42</td></tr>
    <tr><td style="text-align:left;">Max Score</td><td style="text-align:center;">85.77</td><td style="text-align:center;">92.81</td><td style="text-align:center;">94.33</td><td style="text-align:center;">1.52</td><td style="text-align:center;">8.56</td></tr>
    <tr><td style="text-align:left;">Std Dev</td><td style="text-align:center;">15.12</td><td style="text-align:center;">11.87</td><td style="text-align:center;">13.17</td><td style="text-align:center;">1.30</td><td style="text-align:center;">-1.95</td></tr>
  </tbody>
</table>

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Evaluator</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer + FS Examples</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer + FS + CoT</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (FS: with vs. without CoT)</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (FS + CoT vs. Benchmark)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left;">GPT-5</td><td style="text-align:center;">53.33</td><td style="text-align:center;">55.43</td><td style="text-align:center;">63.39</td><td style="text-align:center;">7.96</td><td style="text-align:center;">10.06</td></tr>
    <tr><td style="text-align:left;">Claude Sonnet 4</td><td style="text-align:center;">76.67</td><td style="text-align:center;">84.06</td><td style="text-align:center;">85.71</td><td style="text-align:center;">1.65</td><td style="text-align:center;">9.04</td></tr>
    <tr><td style="text-align:left;">Gemini 2.5 Pro</td><td style="text-align:center;">53.84</td><td style="text-align:center;">67.37</td><td style="text-align:center;">67.47</td><td style="text-align:center;">0.10</td><td style="text-align:center;">13.63</td></tr>
  </tbody>
</table>

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Metric</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer + FS Examples</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer + FS + CoT</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (FS: with vs. without CoT)</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (FS + CoT vs. Benchmark)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left;">Tone</td><td style="text-align:center;">90.19</td><td style="text-align:center;">89.73</td><td style="text-align:center;">89.61</td><td style="text-align:center;">-0.12</td><td style="text-align:center;">-0.58</td></tr>
    <tr><td style="text-align:left;">Clarity</td><td style="text-align:center;">79.37</td><td style="text-align:center;">85.01</td><td style="text-align:center;">85.62</td><td style="text-align:center;">0.61</td><td style="text-align:center;">6.25</td></tr>
    <tr><td style="text-align:left;">Length</td><td style="text-align:center;">79.13</td><td style="text-align:center;">82.39</td><td style="text-align:center;">82.72</td><td style="text-align:center;">0.33</td><td style="text-align:center;">3.59</td></tr>
    <tr><td style="text-align:left;">Structure</td><td style="text-align:center;">43.80</td><td style="text-align:center;">40.82</td><td style="text-align:center;">41.08</td><td style="text-align:center;">0.26</td><td style="text-align:center;">-2.72</td></tr>
    <tr><td style="text-align:left;">Accuracy (% no hallucinations)</td><td style="text-align:center;">72.7%</td><td style="text-align:center;">78.7%</td><td style="text-align:center;">80.0%</td><td style="text-align:center;">1.3%</td><td style="text-align:center;">7.3%</td></tr>
    <tr><td style="text-align:left;">Completeness (% complete)</td><td style="text-align:center;">38.7%</td><td style="text-align:center;">62.7%</td><td style="text-align:center;">62.7%</td><td style="text-align:center;">0.0%</td><td style="text-align:center;">24.0%</td></tr>
    <tr><td style="text-align:left;">Consistency (% no issues)</td><td style="text-align:center;">60.7%</td><td style="text-align:center;">60.0%</td><td style="text-align:center;">71.3%</td><td style="text-align:center;">11.3%</td><td style="text-align:center;">10.6%</td></tr>
  </tbody>
</table>

</div>

Somewhat surprisingly, adding these four words generally seemed to result in a much higher score - the mean improved by 3 points and median by almost 8 points. Remarkably, consistency improved by 11 percentage points compared to the previous run without CoT prompting. The generally increased performance however seemed to come with slightly higher diversity of scores, with standard deviation being a little higher and min-max score spread higher. This intuitively could make sense, as step-by-step prompting may amplify both strong and weak reasoning, increasing variance.

### System Parameter

Moving role and constraint instructions from the user prompt to the system parameter did not materially improve performance and resulted in a lower median score. As overall results were comparable or slightly worse, this change was not retained.

<div style="display:flex; gap:20px; align-items:flex-start; font-size:90%;">

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Metric</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer + FS + CoT</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer + FS + CoT + system</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (with vs. without system)</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (with system vs. Benchmark)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left;">Mean Score</td><td style="text-align:center;">61.28</td><td style="text-align:center;">72.19</td><td style="text-align:center;">69.33</td><td style="text-align:center;">-2.86</td><td style="text-align:center;">8.05</td></tr>
    <tr><td style="text-align:left;">Median Score</td><td style="text-align:center;">63.58</td><td style="text-align:center;">76.51</td><td style="text-align:center;">69.11</td><td style="text-align:center;">-7.40</td><td style="text-align:center;">5.53</td></tr>
    <tr><td style="text-align:left;">Min Score</td><td style="text-align:center;">26.85</td><td style="text-align:center;">35.27</td><td style="text-align:center;">34.63</td><td style="text-align:center;">-0.64</td><td style="text-align:center;">7.78</td></tr>
    <tr><td style="text-align:left;">Max Score</td><td style="text-align:center;">85.77</td><td style="text-align:center;">94.33</td><td style="text-align:center;">94.07</td><td style="text-align:center;">-0.26</td><td style="text-align:center;">8.30</td></tr>
    <tr><td style="text-align:left;">Std Dev</td><td style="text-align:center;">15.12</td><td style="text-align:center;">13.17</td><td style="text-align:center;">15.38</td><td style="text-align:center;">2.21</td><td style="text-align:center;">0.26</td></tr>
  </tbody>
</table>

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Evaluator</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer + FS + CoT</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer + FS + CoT + system</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (with vs. without system)</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (with system vs. Benchmark)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left;">GPT-5</td><td style="text-align:center;">53.33</td><td style="text-align:center;">63.39</td><td style="text-align:center;">61.17</td><td style="text-align:center;">-2.22</td><td style="text-align:center;">7.84</td></tr>
    <tr><td style="text-align:left;">Claude Sonnet 4</td><td style="text-align:center;">76.67</td><td style="text-align:center;">85.71</td><td style="text-align:center;">81.59</td><td style="text-align:center;">-4.12</td><td style="text-align:center;">4.92</td></tr>
    <tr><td style="text-align:left;">Gemini 2.5 Pro</td><td style="text-align:center;">53.84</td><td style="text-align:center;">67.47</td><td style="text-align:center;">65.24</td><td style="text-align:center;">-2.23</td><td style="text-align:center;">11.40</td></tr>
  </tbody>
</table>

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Metric</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer + FS + CoT</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer + FS + CoT + system</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (with vs. without system)</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (with system vs. Benchmark)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left;">Tone</td><td style="text-align:center;">90.19</td><td style="text-align:center;">89.61</td><td style="text-align:center;">89.89</td><td style="text-align:center;">0.28</td><td style="text-align:center;">-0.30</td></tr>
    <tr><td style="text-align:left;">Clarity</td><td style="text-align:center;">79.37</td><td style="text-align:center;">85.62</td><td style="text-align:center;">84.52</td><td style="text-align:center;">-1.10</td><td style="text-align:center;">5.15</td></tr>
    <tr><td style="text-align:left;">Length</td><td style="text-align:center;">79.13</td><td style="text-align:center;">82.72</td><td style="text-align:center;">82.13</td><td style="text-align:center;">-0.59</td><td style="text-align:center;">3.00</td></tr>
    <tr><td style="text-align:left;">Structure</td><td style="text-align:center;">43.80</td><td style="text-align:center;">41.08</td><td style="text-align:center;">40.55</td><td style="text-align:center;">-0.53</td><td style="text-align:center;">-3.25</td></tr>
    <tr><td style="text-align:left;">Accuracy (% no hallucinations)</td><td style="text-align:center;">72.7%</td><td style="text-align:center;">80.0%</td><td style="text-align:center;">77.9%</td><td style="text-align:center;">-2.1%</td><td style="text-align:center;">5.2%</td></tr>
    <tr><td style="text-align:left;">Completeness (% complete)</td><td style="text-align:center;">38.7%</td><td style="text-align:center;">62.7%</td><td style="text-align:center;">61.3%</td><td style="text-align:center;">-1.4%</td><td style="text-align:center;">22.6%</td></tr>
    <tr><td style="text-align:left;">Consistency (% no issues)</td><td style="text-align:center;">60.7%</td><td style="text-align:center;">71.3%</td><td style="text-align:center;">64.0%</td><td style="text-align:center;">-7.3%</td><td style="text-align:center;">3.3%</td></tr>
  </tbody>
</table>

</div>

### XML Tags

Introducing XML tags to delineate examples and inputs slightly improved minimum scores but reduced mean and median performance. Given the overall degradation and added prompt complexity, XML structuring was not retained.

<div style="display:flex; gap:20px; align-items:flex-start; font-size:90%;">

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Metric</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer + FS + CoT</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer + FS + CoT + XML</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (with vs. without XML)</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (with XML vs. Benchmark)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left;">Mean Score</td><td style="text-align:center;">61.28</td><td style="text-align:center;">72.19</td><td style="text-align:center;">66.5</td><td style="text-align:center;">-5.69</td><td style="text-align:center;">5.22</td></tr>
    <tr><td style="text-align:left;">Median Score</td><td style="text-align:center;">63.58</td><td style="text-align:center;">76.51</td><td style="text-align:center;">67.79</td><td style="text-align:center;">-8.72</td><td style="text-align:center;">4.21</td></tr>
    <tr><td style="text-align:left;">Min Score</td><td style="text-align:center;">26.85</td><td style="text-align:center;">35.27</td><td style="text-align:center;">42.46</td><td style="text-align:center;">7.19</td><td style="text-align:center;">15.61</td></tr>
    <tr><td style="text-align:left;">Max Score</td><td style="text-align:center;">85.77</td><td style="text-align:center;">94.33</td><td style="text-align:center;">94.02</td><td style="text-align:center;">-0.31</td><td style="text-align:center;">8.25</td></tr>
    <tr><td style="text-align:left;">Std Dev</td><td style="text-align:center;">15.12</td><td style="text-align:center;">13.17</td><td style="text-align:center;">12.43</td><td style="text-align:center;">-0.74</td><td style="text-align:center;">-2.69</td></tr>
  </tbody>
</table>

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Evaluator</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer + FS + CoT</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer + FS + CoT + XML</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (with vs. without XML)</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (with XML vs. Benchmark)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left;">GPT-5</td><td style="text-align:center;">53.33</td><td style="text-align:center;">63.39</td><td style="text-align:center;">54.87</td><td style="text-align:center;">-8.52</td><td style="text-align:center;">1.54</td></tr>
    <tr><td style="text-align:left;">Claude Sonnet 4</td><td style="text-align:center;">76.67</td><td style="text-align:center;">85.71</td><td style="text-align:center;">80.08</td><td style="text-align:center;">-5.63</td><td style="text-align:center;">3.41</td></tr>
    <tr><td style="text-align:left;">Gemini 2.5 Pro</td><td style="text-align:center;">53.84</td><td style="text-align:center;">67.47</td><td style="text-align:center;">64.05</td><td style="text-align:center;">-3.42</td><td style="text-align:center;">10.21</td></tr>
  </tbody>
</table>

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Metric</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer + FS + CoT</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer + FS + CoT + XML</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (with vs. without XML)</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (with XML vs. Benchmark)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left;">Tone</td><td style="text-align:center;">90.19</td><td style="text-align:center;">89.61</td><td style="text-align:center;">88.63</td><td style="text-align:center;">-0.98</td><td style="text-align:center;">-1.56</td></tr>
    <tr><td style="text-align:left;">Clarity</td><td style="text-align:center;">79.37</td><td style="text-align:center;">85.62</td><td style="text-align:center;">84.83</td><td style="text-align:center;">-0.79</td><td style="text-align:center;">5.46</td></tr>
    <tr><td style="text-align:left;">Length</td><td style="text-align:center;">79.13</td><td style="text-align:center;">82.72</td><td style="text-align:center;">77.53</td><td style="text-align:center;">-5.19</td><td style="text-align:center;">-1.60</td></tr>
    <tr><td style="text-align:left;">Structure</td><td style="text-align:center;">43.80</td><td style="text-align:center;">41.08</td><td style="text-align:center;">41.11</td><td style="text-align:center;">0.03</td><td style="text-align:center;">-2.69</td></tr>
    <tr><td style="text-align:left;">Accuracy (% no hallucinations)</td><td style="text-align:center;">72.7%</td><td style="text-align:center;">80.0%</td><td style="text-align:center;">76.5%</td><td style="text-align:center;">-3.5%</td><td style="text-align:center;">3.8%</td></tr>
    <tr><td style="text-align:left;">Completeness (% complete)</td><td style="text-align:center;">38.7%</td><td style="text-align:center;">62.7%</td><td style="text-align:center;">59.7%</td><td style="text-align:center;">-3.0%</td><td style="text-align:center;">21.0%</td></tr>
    <tr><td style="text-align:left;">Consistency (% no issues)</td><td style="text-align:center;">60.7%</td><td style="text-align:center;">71.3%</td><td style="text-align:center;">56.4%</td><td style="text-align:center;">-14.9%</td><td style="text-align:center;">-4.3%</td></tr>
  </tbody>
</table>

</div>

### Iterative Refinement

Independent iterative refinement, where Claude revised memos based on evaluator feedback, consistently degraded performance across rounds, despite a slight reduction in variance. Given higher runtime and no quality gains, this approach was not retained. <br>
In this setup, each memo underwent three refinement processes (one for each evaluator):
1. Claude Sonnet 4 generated an initial memo.
2. The chosen evaluator evaluated the memo using the standard evaluation prompt.
3. The evaluator’s feedback, along with the original prompt, source credit agreement, and produced memo was fed back to Claude as input for revision (“You are refining an investment memo based on evaluation feedback [...]”)
4. Steps 2 and 3 were potentially repeated an additional n times based on the chosen number of refinement rounds
5. Claude produced a final memo, which was then evaluated by all three judges (GPT-5, Claude, Gemini).

This setup was run with 2 refinement rounds (so a total of 3 memo generation attempts per input, per evaluator), with results illustrated below:

<div style="text-align:center;">
  <img src="charts/fig1.png" width="700"/>
</div>

<div style="text-align:center;">
  <img src="charts/fig2.png" width="700"/>
</div>

<div style="text-align:center;">
  <img src="charts/fig3.png" width="700"/>
</div>

Providing combined feedback from all evaluators produced similar or worse degradation than independent refinement, while increasing prompt length and complexity. The “combined” approach used all three evaluators to provide feedback simultaneously:
1. Claude generated an initial memo.
2. GPT-5, Claude, and Gemini each produced evaluation feedback.
3. All three feedback summaries were concatenated and fed back to Claude.
4. Steps 2 and 3 were potentially repeated an additional n times based on the chosen number of refinement rounds
5. Claude produced a final memo, which was then evaluated by all three judges (GPT-5, Claude, Gemini).

This setup was also run with two refinement rounds (3 memo generations total per input), with results presented below. Despite faster runtime than the independent variant, this approach did not seem to improve quality and was not retained. 

<div style="text-align:center;">
  <img src="charts/fig4.png" width="700"/>
</div>

<div style="text-align:center;">
  <img src="charts/fig5.png" width="700"/>
</div>

<div style="text-align:center;">
  <img src="charts/fig6.png" width="700"/>
</div>

Overall, adding iterative refinement rounds largely increased run time, and did not seem to provide material improvements to performance. A few feasible reasons for this could be:

- Model evaluations are **noisy and weakly aligned** with the scoring objective, and small differences in phrasing, focus, or hallucinated issues get compounded at each round (or at best, do not provide improvements).
- **Anchoring**: even if the feedback is correct, because the model is being fed an already-written memo and asked to edit it, there is little remaining headroom
- **Conflicting guidance** in the “combined” setting: the three evaluators sometimes disagree or emphasize different issues, in which case Claude has to satisfy partially conflicting instructions, which encourages generic text 
- The combined prompt becomes **very long and complex**, and perhaps some instructions get ignored or de-prioritized
- Measurement noise and small deltas: judge scores themselves have a variance - the deterioration per round is small, and hard to distinguish from noise that could have just appeared as a result of **repeated runs** - for example, differences across refinement rounds were comparable in magnitude to run-to-run variance, suggesting results could have just been driven by measurement noise.

This supports the interpretation that the base prompt + FS + CoT already captures most of the achievable performance under this setup, and extra refinement mainly increases runtime and complexity.

### Other Techniques Considered

Several techniques were considered but not applied, including response prefilling, long-context features, extended thinking, and prompt chaining. These were deemed unnecessary or inefficient given that documents fit within standard context limits, the task involved single-document summarization, and core performance issues related to interpretation rather than retrieval or multi-stage reasoning.

## <span style="color:darkred;">5. Test Set Validation</span>

In light of the performance outlined above and in the graph below, the chosen prompting technique is the OpenAI cookbook-optimized prompt, with chain-of-thought prompting and few shot examples.

**Training Set**

<div style="display:flex; gap:20px; align-items:flex-start; font-size:90%;">

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Metric</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer + FS + CoT</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">% Change</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left;">Mean Score</td><td style="text-align:center;">61.28</td><td style="text-align:center;">72.19</td><td style="text-align:center;">10.91</td><td style="text-align:center;">17.8%</td></tr>
    <tr><td style="text-align:left;">Median Score</td><td style="text-align:center;">63.58</td><td style="text-align:center;">76.51</td><td style="text-align:center;">12.93</td><td style="text-align:center;">20.3%</td></tr>
    <tr><td style="text-align:left;">Min Score</td><td style="text-align:center;">26.85</td><td style="text-align:center;">35.27</td><td style="text-align:center;">8.42</td><td style="text-align:center;">31.4%</td></tr>
    <tr><td style="text-align:left;">Max Score</td><td style="text-align:center;">85.77</td><td style="text-align:center;">94.33</td><td style="text-align:center;">8.56</td><td style="text-align:center;">10.0%</td></tr>
    <tr><td style="text-align:left;">Std Dev</td><td style="text-align:center;">15.12</td><td style="text-align:center;">13.17</td><td style="text-align:center;">-1.95</td><td style="text-align:center;">-12.9%</td></tr>
  </tbody>
</table>

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Evaluator</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer + FS + CoT</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">% Change</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left;">GPT-5</td><td style="text-align:center;">53.33</td><td style="text-align:center;">63.39</td><td style="text-align:center;">10.06</td><td style="text-align:center;">18.9%</td></tr>
    <tr><td style="text-align:left;">Claude Sonnet 4</td><td style="text-align:center;">76.67</td><td style="text-align:center;">85.71</td><td style="text-align:center;">9.04</td><td style="text-align:center;">11.8%</td></tr>
    <tr><td style="text-align:left;">Gemini 2.5 Pro</td><td style="text-align:center;">53.84</td><td style="text-align:center;">67.47</td><td style="text-align:center;">13.63</td><td style="text-align:center;">25.3%</td></tr>
  </tbody>
</table>

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Metric</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer + FS + CoT</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">% Change</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left;">Tone</td><td style="text-align:center;">90.19</td><td style="text-align:center;">89.61</td><td style="text-align:center;">-0.58</td><td style="text-align:center;">-0.6%</td></tr>
    <tr><td style="text-align:left;">Clarity</td><td style="text-align:center;">79.37</td><td style="text-align:center;">85.62</td><td style="text-align:center;">6.25</td><td style="text-align:center;">7.9%</td></tr>
    <tr><td style="text-align:left;">Length</td><td style="text-align:center;">79.13</td><td style="text-align:center;">82.72</td><td style="text-align:center;">3.59</td><td style="text-align:center;">4.5%</td></tr>
    <tr><td style="text-align:left;">Structure</td><td style="text-align:center;">43.80</td><td style="text-align:center;">41.08</td><td style="text-align:center;">-2.72</td><td style="text-align:center;">-6.2%</td></tr>
    <tr><td style="text-align:left;">Accuracy (% no hallucinations)</td><td style="text-align:center;">72.7%</td><td style="text-align:center;">80.0%</td><td style="text-align:center;">0.07</td><td style="text-align:center;">10.0%</td></tr>
    <tr><td style="text-align:left;">Completeness (% complete)</td><td style="text-align:center;">38.7%</td><td style="text-align:center;">62.7%</td><td style="text-align:center;">0.24</td><td style="text-align:center;">62.0%</td></tr>
    <tr><td style="text-align:left;">Consistency (% no issues)</td><td style="text-align:center;">60.7%</td><td style="text-align:center;">71.3%</td><td style="text-align:center;">0.11</td><td style="text-align:center;">17.5%</td></tr>
  </tbody>
</table>

</div>

<div style="text-align:center;">
  <img src="charts/fig7.png" width="500"/>
</div>

This prompting technique resulted in a 10.9 percentage point, or 17.8% increase, in the overall mean score compared to the benchmark, a 31.4% increase in the lowest score of the dataset, and a 20.3% increase in the median score. In order to assess the generalizability of this performance improvement, this prompting technique and initial benchmark were run on the test set. Given compute resources, our training set was significantly smaller (50 inputs) than the testing set (446 inputs, as it excluded the three examples used for few-shot examples).

**Test Set**

<div style="display:flex; gap:20px; align-items:flex-start; font-size:90%;">

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Metric</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer + FS + CoT</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">% Change</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left;">Mean Score</td><td style="text-align:center;">65.46</td><td style="text-align:center;">65.86</td><td style="text-align:center;">0.40</td><td style="text-align:center;">0.6%</td></tr>
    <tr><td style="text-align:left;">Median Score</td><td style="text-align:center;">68.01</td><td style="text-align:center;">68.28</td><td style="text-align:center;">0.27</td><td style="text-align:center;">0.4%</td></tr>
    <tr><td style="text-align:left;">Min Score</td><td style="text-align:center;">25.79</td><td style="text-align:center;">17.52</td><td style="text-align:center;">-8.27</td><td style="text-align:center;">-32.1%</td></tr>
    <tr><td style="text-align:left;">Max Score</td><td style="text-align:center;">94.75</td><td style="text-align:center;">94.38</td><td style="text-align:center;">-0.37</td><td style="text-align:center;">-0.4%</td></tr>
    <tr><td style="text-align:left;">Std Dev</td><td style="text-align:center;">13.63</td><td style="text-align:center;">13.82</td><td style="text-align:center;">0.19</td><td style="text-align:center;">1.4%</td></tr>
  </tbody>
</table>

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Evaluator</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer + FS + CoT</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">% Change</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left;">GPT-5</td><td style="text-align:center;">56.16</td><td style="text-align:center;">55.81</td><td style="text-align:center;">-0.35</td><td style="text-align:center;">-0.6%</td></tr>
    <tr><td style="text-align:left;">Claude Sonnet 4</td><td style="text-align:center;">76.56</td><td style="text-align:center;">80.62</td><td style="text-align:center;">4.06</td><td style="text-align:center;">5.3%</td></tr>
    <tr><td style="text-align:left;">Gemini 2.5 Pro</td><td style="text-align:center;">63.65</td><td style="text-align:center;">61.15</td><td style="text-align:center;">-2.50</td><td style="text-align:center;">-3.9%</td></tr>
  </tbody>
</table>

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Metric</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer + FS + CoT</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">% Change</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left;">Tone</td><td style="text-align:center;">90.57</td><td style="text-align:center;">89.50</td><td style="text-align:center;">-1.07</td><td style="text-align:center;">-1.2%</td></tr>
    <tr><td style="text-align:left;">Clarity</td><td style="text-align:center;">82.12</td><td style="text-align:center;">85.25</td><td style="text-align:center;">3.13</td><td style="text-align:center;">3.8%</td></tr>
    <tr><td style="text-align:left;">Length</td><td style="text-align:center;">74.22</td><td style="text-align:center;">81.40</td><td style="text-align:center;">7.18</td><td style="text-align:center;">9.7%</td></tr>
    <tr><td style="text-align:left;">Structure</td><td style="text-align:center;">44.37</td><td style="text-align:center;">40.93</td><td style="text-align:center;">-3.44</td><td style="text-align:center;">-7.8%</td></tr>
    <tr><td style="text-align:left;">Accuracy (% no hallucinations)</td><td style="text-align:center;">78.6%</td><td style="text-align:center;">76.6%</td><td style="text-align:center;">-0.02</td><td style="text-align:center;">-2.5%</td></tr>
    <tr><td style="text-align:left;">Completeness (% complete)</td><td style="text-align:center;">55.8%</td><td style="text-align:center;">54.0%</td><td style="text-align:center;">-0.02</td><td style="text-align:center;">-3.2%</td></tr>
    <tr><td style="text-align:left;">Consistency (% no issues)</td><td style="text-align:center;">54.6%</td><td style="text-align:center;">58.6%</td><td style="text-align:center;">0.04</td><td style="text-align:center;">7.3%</td></tr>
  </tbody>
</table>

</div>

Overall, there was no meaningful performance improvement on the test set relative to the benchmark prompt, indicating that the optimized prompt did not generalize. Several factors may explain this:
- The optimization loop relied on only 50 training examples, which could have been too few to reliably detect true performance gains. With such a **small sample**, results are highly sensitive to outliers and random variance, making any “best” prompt selection unstable.
- Training-test **distribution shift**: the 50 training examples may not have been representative of the full dataset (e.g., in transaction type, drafting style, length, or complexity). As a result, the optimized prompt may have adapted to patterns specific to the training subset that do not hold in the 446-case test set.
- **Prompt overfitting** from iterative tuning: repeatedly refining the prompt on the same small set effectively overfits it to the quirks of those examples, similar to model overfitting. This reduces its ability to generalize.
- If the benchmark prompt is already quite good, then the **remaining “headroom” for improvement is small** and perhaps optimizing on a small slice of data mainly captures noise.

## <span style="color:darkred;">6. Conclusion and Key Contributions</span>


This project shows that while prompt engineering can produce meaningful in-sample gains, those gains do not reliably generalize when optimization is performed on small, noisy datasets. The primary contribution is therefore not a ‘best prompt,’ but an evaluation framework that exposes this limitation clearly.

### Key Contributions

**End-to-end, evaluation-driven memo generation pipeline:** This project builds a complete, practical pipeline for generating private credit investment memos directly from raw legal agreements. It ingests real SEC-filed credit agreements, applies automated cleaning and normalization, and produces IC-style memos using frontier LLMs. The system operates on realistic documents and outputs artifacts that closely resemble what analysts produce in practice, grounding the research in a real investment workflow.


**Scalable, multi-model evaluation harness tailored to private credit:** Another contribution is the design of a reusable evaluation framework specifically aligned with private-credit memo quality. The harness operationalizes otherwise subjective concepts (accuracy, completeness, consistency, and presentation quality) using a multi-model LLM-consensus approach and batch execution for scalability. This enables statistically analyzable comparisons across prompts and models at reasonable cost, and surfaces evaluator disagreement and self-agreement bias as first-order considerations.

**Systematic “reverse ablation” study of prompt-engineering techniques:** The project conducts a controlled, component-by-component study of prompt optimization techniques on a single frontier model (Claude Sonnet 4). Techniques evaluated include Anthropic and OpenAI prompt optimizers, meta-prompting, added context, few-shot examples, chain-of-thought prompting, system-prompt relocation, XML structuring, and multiple variants of iterative refinement. By incrementally adding and removing techniques, the study isolates which interventions matter for this task and which add complexity without measurable benefit. <br> Results suggest that relatively simple techniques (particularly few-shot prompting and chain-of-thought) drive the largest gains in memo quality, while others (XML tags, system-parameter relocation, iterative refinement) provide little or no benefit and often increase runtime or instability. On the training set, these methods produced an ~18% improvement in mean score, but the failure to generalize to a larger test set highlights the risk of prompt overfitting when optimization is performed on small, noisy samples.

**Evidence of performance stability and limited headroom in frontier models:** Across dozens of prompt variants and repeated runs, scores remained within a relatively narrow band, suggesting that frontier models already perform this task with moderate consistency. Improvements tended to shift the distribution rather than transform it, implying limited headroom under a pure prompt-engineering regime. This stability itself is a useful finding: it suggests that marginal gains from further prompt tweaking are likely small compared to gains from better data, stronger supervision, or model-level adaptation.

**Business-relevant framing of impact and limitations:** Although the system is not production-ready, the paper paves way for model-quality improvements which could have business-level impacts. Even a partial (~18%) improvement in memo quality could materially reduce analyst time spent on first-pass drafting, shifting effort toward review and judgment. In a screening-heavy private-credit workflow, this could increase the number of deals screened by up to 4x (if, say, the AI tool is 75% accurate and allows for 75% time-saving), thereby increasing the ultimate number of deals closed by reducing the risk of prematurely discarding viable opportunities due to a lack of capacity on a deal team. Given the scale of deals usually closed by large funds, this can result in an impact of the order of millions of dollars.

**Why prompt optimization alone is insufficient:** Finally, the project demonstrates that prompt optimization must be treated as use-case-specific and evaluation-dependent, not as a one-time “best prompt” to be reused broadly. The lack of test-set generalization underscores the need for larger and more representative datasets, stronger evaluation signals, and eventually fine-tuning or human-in-the-loop feedback. This work serves as a foundation for more robust, finance-specific LLM systems.

### Further Work

While this project did result in some performance improvement, several limitations surfaced across data, evaluation, and modeling design. The potential extensions below could substantially strengthen both the system and the evaluation framework.

**Data**:
Although approximately 500 credit agreements were collected, only a small subset was used for prompt optimization due to cost constraints. With limited sample sizes, run-level statistics (means and variances) are indicative but insufficient for strong claims about generalization. Using a larger training set would likely reduce noise, mitigate prompt overfitting, and improve robustness. One approach to obtain one could have been constructing a template credit agreement/template memo with placeholders for key economic terms (rate, leverage thresholds, covenants) and automatically generating many synthetic variations by sampling plausible numerical values. <br>
The dataset was also drawn from a single public source and reflects recent (2023 onwards), large-cap transactions, limiting coverage of private-company deals, older agreements, and alternative drafting conventions.This introduces potential sampling bias: the system may perform well on similar, high-quality agreements but may generalize less effectively to private-company or older contracts with different drafting conventions. A more diverse dataset could help with this. <br>
Another way to obtain high quality data/signals (for the ‘iterative refinement’ case) would be to incorporate human-in-the-loop feedback. An interface allowing analysts to rate generated memos could be used to refine prompts, calibrate LLM-judge scores, or eventually train a reward model. 

**Evaluation Design:** The evaluation framework relies on model-graded judgments, which introduces noise: evaluators can hallucinate issues, disagree on borderline cases, or diverge from human analyst priorities. Increasing the number or diversity of evaluators could reduce variance and improve reliability. In addition, the current scoring scheme weights all metrics equally and does not sharply distinguish between hallucinations (false positives) and omissions (false negatives), despite hallucinations being more damaging in practice. Introducing asymmetric penalties, or explicitly tracking type I versus type II errors, could better align evaluation outcomes with real-world risk tolerance. <br> Additionally, incorporating a measure of latency and/or cost in the general evaluation score would yield a more comprehensive evaluation harness.

**Tool-Augmented and Agentic Systems:** Future work could explore lightweight agentic extensions to the pipeline. For example, an agent could decide when to call external tools (retrieval, company background lookup, news search) to augment memos with contextual information not present in the agreement itself. The task could also be decomposed into explicit subtasks (extract → verify → contextualize → write), with intermediate traces logged for debugging and evaluation. <br>
Using citation-capable APIs to generate memos with inline references to specific agreement passages could further improve accuracy. These citations could be programmatically verified via string or semantic matching, enabling a more objective accuracy metric than LLM consensus alone.

**Model Scope and Generalization:** This project focused on prompt optimization for a single model (Claude Sonnet 4). Future work could apply the same evaluation-driven methodology to other frontier models, or explore multi-model ensembles that blend outputs to reduce model-specific failure modes. Finally, generalization could be tested more rigorously by applying the pipeline to adjacent but structurally different document types, such as term sheets or indentures, which contain similar economic content but follow different conventions. Performance on these transfers would provide a stronger signal of whether the system has learned generalizable structure rather than document-specific patterns.
